# Logistic Regression: Original BoL Feature Analysis

This notebook fits the L2 logistic regression tabular baseline using the original BoL feature index and the person-specific cutoff logic.

It contains the full analysis in one place:

1. Load target and original BoL features
2. Preprocess features
3. Fit L2 logistic regression
4. Evaluate model performance
5. Save predictions, coefficients, and metrics
6. **Feature Importance and Linear SHAP-style Attributions**

The feature-importance/XAI section is in this same notebook, directly after the output-saving section, matching the structure of the gradient-boosting notebook.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


def find_project_dir(start=None):
    """Find the project root independent of where Jupyter was launched."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        has_raw_data = (candidate / "nlsy79_child_youngadult_selected_crime_features.csv").exists()
        has_tabular_dir = (candidate / "Tabular_based_models").exists()
        has_bol_dir = (candidate / "BoL approach").exists()
        if has_raw_data and has_tabular_dir and has_bol_dir:
            return candidate
    raise FileNotFoundError(
        "Could not find project root. Start Jupyter inside the Bol_Crime project folder "
        "or one of its subfolders."
    )


PROJECT_DIR = find_project_dir()
TABULAR_DIR = PROJECT_DIR / "Tabular_based_models"
MODEL_DIR = TABULAR_DIR / "logistic_regression"

DATA_PATH = PROJECT_DIR / "nlsy79_child_youngadult_selected_crime_features.csv"
FEATURE_INDEX_PATH = PROJECT_DIR / "BoL approach" / "metadata_examples" / "child_crime_broad_persistent_feature_index.csv"
TARGETS_PATH = TABULAR_DIR / "data" / "targets" / "nlsy79_temporal_delinquency_targets_2000_2020.csv"
OUT_DIR = MODEL_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_PATHS = {
    "raw data": DATA_PATH,
    "original BoL feature index": FEATURE_INDEX_PATH,
    "target csv": TARGETS_PATH,
}
missing_paths = {name: path for name, path in REQUIRED_PATHS.items() if not path.exists()}
if missing_paths:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(f"{name}: {path}" for name, path in missing_paths.items()))

TARGET = "later_persistent_delinquency_contact_2000_2020"
SECOND_EVENT_YEAR = "later_delinquency_contact_second_event_year_2000_2020"
LAST_OBSERVED_YEAR = "later_delinquency_contact_last_observed_year_2000_2020"
MISSING_CODES = {-1, -2, -3, -4, -5, -7}
RANDOM_SEED = 2026
TEST_SIZE = 0.30
FEATURE_SOURCE = "original BoL broad persistent feature index"
CUTOFF_RULE = "positive: before second event year; negative: before last observed target year"

print("Project dir:", PROJECT_DIR)
print("Target file:", TARGETS_PATH)


## Load Original BoL Features and Target

We use the original 864-feature BoL index and the temporally defined persistent delinquency/contact target.

In [ ]:
data = pd.read_csv(DATA_PATH)
feature_index = pd.read_csv(FEATURE_INDEX_PATH)
targets = pd.read_csv(TARGETS_PATH)

feature_cols = [c for c in feature_index["csv_code"].tolist() if c in data.columns]
feature_year = feature_index.set_index("csv_code")["survey_year"].to_dict()

model_df = data[["C0000100"] + feature_cols].merge(
    targets[["C0000100", TARGET, SECOND_EVENT_YEAR, LAST_OBSERVED_YEAR]],
    on="C0000100",
    how="inner",
)
model_df = model_df[model_df[TARGET].notna()].copy()
model_df["cutoff_year"] = np.where(
    model_df[TARGET].eq(1),
    model_df[SECOND_EVENT_YEAR],
    model_df[LAST_OBSERVED_YEAR],
)
model_df = model_df[model_df["cutoff_year"].notna()].copy()

print("Rows:", len(model_df))
print("Original BoL features available:", len(feature_cols))
print("Target balance:")
print(model_df[TARGET].value_counts().sort_index())
print("Base rate:", round(model_df[TARGET].mean(), 3))
print("Cutoff rule:", CUTOFF_RULE)

feature_index[["csv_code", "ref_id", "variable", "survey_year", "feature_group", "question"]].head(10)


## Preprocessing

Negative NLSY missing codes are recoded to NA. Features at or after each person's cutoff year are also set to NA. Missing values are then median-imputed using the training set only. Logistic regression additionally standardizes the features.

In [ ]:
def clean_feature_matrix_with_cutoff(df, feature_cols, feature_year):
    x = df[feature_cols].copy()
    cutoff = pd.to_numeric(df["cutoff_year"], errors="coerce")

    for col in feature_cols:
        x[col] = pd.to_numeric(x[col], errors="coerce")
        x[col] = x[col].replace([np.inf, -np.inf], np.nan)
        x.loc[x[col].isin(MISSING_CODES), col] = np.nan

        year = feature_year.get(col)
        if str(year) != "XRND":
            year_num = pd.to_numeric(pd.Series([year]), errors="coerce").iloc[0]
            if pd.notna(year_num):
                # Mirror original BoL: only information before the person-specific cutoff is allowed.
                x.loc[cutoff <= year_num, col] = np.nan
    return x


def train_test_split_stratified(y, test_size=0.30, seed=2026):
    rng = np.random.default_rng(seed)
    train_idx = []
    test_idx = []
    for label in sorted(np.unique(y)):
        idx = np.where(y == label)[0]
        rng.shuffle(idx)
        n_test = int(round(len(idx) * test_size))
        test_idx.extend(idx[:n_test].tolist())
        train_idx.extend(idx[n_test:].tolist())
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)
    return np.array(train_idx), np.array(test_idx)


def fit_median_imputer(x_train):
    return x_train.median(axis=0, skipna=True).fillna(0.0)


def impute_with_median(x, medians):
    arr = x.fillna(medians).to_numpy(dtype=float)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)


def fit_standardizer(x):
    means = x.mean(axis=0)
    stds = x.std(axis=0)
    stds[stds == 0] = 1.0
    return means, stds


def standardize(x, means, stds):
    z = (x - means) / stds
    return np.clip(np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0), -10.0, 10.0)


def sigmoid(z):
    z = np.clip(z, -35, 35)
    return 1.0 / (1.0 + np.exp(-z))


def auc_score(y_true, prob):
    order = np.argsort(prob)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(prob) + 1)
    pos = y_true == 1
    n_pos = int(pos.sum())
    n_neg = int((~pos).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    rank_sum_pos = ranks[pos].sum()
    return float((rank_sum_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def metrics(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    return {
        "threshold": threshold,
        "accuracy": float((pred == y_true).mean()),
        "auc": auc_score(y_true, prob),
        "sensitivity_tpr": tp / (tp + fn) if (tp + fn) else np.nan,
        "specificity_tnr": tn / (tn + fp) if (tn + fp) else np.nan,
        "predicted_positive_rate": float(pred.mean()),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


In [ ]:
y = model_df[TARGET].astype(int).to_numpy()
x_raw = clean_feature_matrix_with_cutoff(model_df, feature_cols, feature_year)

train_idx, test_idx = train_test_split_stratified(y, TEST_SIZE, RANDOM_SEED)

x_train_raw = x_raw.iloc[train_idx]
x_test_raw = x_raw.iloc[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

medians = fit_median_imputer(x_train_raw)
x_train_imputed = impute_with_median(x_train_raw, medians)
x_test_imputed = impute_with_median(x_test_raw, medians)

means, stds = fit_standardizer(x_train_imputed)
x_train_std = standardize(x_train_imputed, means, stds)
x_test_std = standardize(x_test_imputed, means, stds)

print("Train N:", len(y_train), "Test N:", len(y_test))
print("Train base rate:", round(y_train.mean(), 3))
print("Test base rate:", round(y_test.mean(), 3))
print("Feature matrix shape:", x_train_imputed.shape)


## Fit L2 Logistic Regression

In [ ]:
L2_STRENGTH = 1.0
LOGIT_LEARNING_RATE = 0.01
LOGIT_MAX_ITER = 5000
LOGIT_TOL = 1e-8


def linear_predict(x_design, beta):
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        z = x_design @ beta
    return np.nan_to_num(z, nan=0.0, posinf=35.0, neginf=-35.0)


def fit_l2_logistic(x, y):
    x_design = np.column_stack([np.ones(len(x)), x])
    beta = np.zeros(x_design.shape[1], dtype=float)
    history = []
    for _ in range(LOGIT_MAX_ITER):
        p = sigmoid(linear_predict(x_design, beta))
        error = p - y
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            grad = (x_design.T @ error) / len(y)
        grad = np.nan_to_num(grad, nan=0.0, posinf=0.0, neginf=0.0)
        grad[1:] += (L2_STRENGTH / len(y)) * beta[1:]
        new_beta = beta - LOGIT_LEARNING_RATE * grad

        eps = 1e-12
        loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
        loss += (L2_STRENGTH / (2 * len(y))) * np.sum(beta[1:] ** 2)
        history.append(float(loss))

        if np.max(np.abs(new_beta - beta)) < LOGIT_TOL:
            beta = new_beta
            break
        beta = new_beta
    return beta, history


def predict_l2_logistic(x, beta):
    x_design = np.column_stack([np.ones(len(x)), x])
    return sigmoid(linear_predict(x_design, beta))

logit_beta, logit_history = fit_l2_logistic(x_train_std, y_train)
logit_train_prob = predict_l2_logistic(x_train_std, logit_beta)
logit_test_prob = predict_l2_logistic(x_test_std, logit_beta)

logit_train_metrics = metrics(y_train, logit_train_prob)
logit_test_metrics = metrics(y_test, logit_test_prob)

print("Iterations:", len(logit_history))
print("Final train loss:", round(logit_history[-1], 4))
print("Test metrics:")
print(pd.Series(logit_test_metrics).to_string())


## Save Outputs

This cell saves predictions, metrics, and the logistic-regression coefficient table. The coefficient table is the first feature-importance view.

The next section, in this same notebook, adds permutation importance and linear SHAP-style XAI.


In [ ]:
test_ids = model_df.iloc[test_idx]["C0000100"].astype(int).to_numpy()

pred_df = pd.DataFrame({
    "C0000100": test_ids,
    "target": TARGET,
    "model": "l2_logistic_regression_original_bol_features_cutoff",
    "y_true": y_test,
    "probability": logit_test_prob,
    "prediction": (logit_test_prob >= 0.5).astype(int),
})
pred_df["correct"] = pred_df["prediction"] == pred_df["y_true"]

coef_df = pd.DataFrame({
    "csv_code": feature_cols,
    "coefficient": logit_beta[1:],
}).merge(
    feature_index[["csv_code", "ref_id", "variable", "survey_year", "feature_group", "question"]],
    on="csv_code",
    how="left",
)
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

metrics_df = pd.DataFrame([
    {"split": "train", **logit_train_metrics},
    {"split": "test", **logit_test_metrics},
])

summary = {
    "target": TARGET,
    "model": "l2_logistic_regression_original_bol_features_cutoff",
    "feature_source": FEATURE_SOURCE,
    "cutoff_rule": CUTOFF_RULE,
    "n_total": int(len(model_df)),
    "n_train": int(len(train_idx)),
    "n_test": int(len(test_idx)),
    "n_features": int(len(feature_cols)),
    "test_size": TEST_SIZE,
    "random_seed": RANDOM_SEED,
    "l2_strength": L2_STRENGTH,
    "learning_rate": LOGIT_LEARNING_RATE,
    "iterations": len(logit_history),
    "final_train_loss": logit_history[-1],
    "train_metrics": logit_train_metrics,
    "test_metrics": logit_test_metrics,
}

pred_df.to_csv(OUT_DIR / "l2_logistic_regression_predictions.csv", index=False)
coef_df.to_csv(OUT_DIR / "l2_logistic_regression_coefficients.csv", index=False)
metrics_df.to_csv(OUT_DIR / "l2_logistic_regression_metrics.csv", index=False)
(OUT_DIR / "l2_logistic_regression_summary.json").write_text(json.dumps(summary, indent=2))

coef_df.head(15)


## Feature Importance and Linear SHAP-style Attributions

This section is part of the same logistic-regression analysis notebook, matching the structure of the gradient-boosting notebook. It explains the fitted model with three complementary views.

1. **Coefficient importance:** ranks standardized features by the absolute logistic-regression coefficient.
2. **Permutation importance:** shuffles one feature in the test set and measures how much AUC/accuracy drops. This is model-performance based.
3. **Linear SHAP-style attributions:** for logistic regression, each standardized feature contributes `coefficient * (value - training_mean)` to the logit. This gives an additive explanation of the prediction around the training baseline and is well suited to a linear model.

Interpretation caution: these are model explanations, not causal effects.


In [ ]:
# Standard coefficient-based importance is already saved in coef_df.
# Here we add test-set permutation importance and linear SHAP-style logit attributions.

def permutation_importance_logistic(x_test, y_test, beta, feature_cols, n_repeats=1, seed=2026):
    rng = np.random.default_rng(seed)
    baseline_prob = predict_l2_logistic(x_test, beta)
    baseline = metrics(y_test, baseline_prob)
    rows = []
    for j, feature in enumerate(feature_cols):
        aucs = []
        accs = []
        for _ in range(n_repeats):
            x_perm = x_test.copy()
            x_perm[:, j] = rng.permutation(x_perm[:, j])
            prob = predict_l2_logistic(x_perm, beta)
            m = metrics(y_test, prob)
            aucs.append(m["auc"])
            accs.append(m["accuracy"])
        rows.append({
            "csv_code": feature,
            "baseline_auc": baseline["auc"],
            "permuted_auc": float(np.mean(aucs)),
            "auc_drop": baseline["auc"] - float(np.mean(aucs)),
            "baseline_accuracy": baseline["accuracy"],
            "permuted_accuracy": float(np.mean(accs)),
            "accuracy_drop": baseline["accuracy"] - float(np.mean(accs)),
        })
    return pd.DataFrame(rows)

perm_importance_df = permutation_importance_logistic(x_test_std, y_test, logit_beta, feature_cols, n_repeats=1, seed=RANDOM_SEED)
perm_importance_df = perm_importance_df.merge(
    feature_index[["csv_code", "ref_id", "variable", "survey_year", "feature_group", "question"]],
    on="csv_code",
    how="left",
).sort_values("auc_drop", ascending=False)

# Linear SHAP-style values in logit space.
train_background_mean = x_train_std.mean(axis=0)
linear_shap_values = (x_test_std - train_background_mean) * logit_beta[1:]
base_logit = float(logit_beta[0] + train_background_mean @ logit_beta[1:])
predicted_logit = linear_predict(np.column_stack([np.ones(len(x_test_std)), x_test_std]), logit_beta)

linear_shap_global_df = pd.DataFrame({
    "csv_code": feature_cols,
    "coefficient": logit_beta[1:],
    "mean_abs_shap_logit": np.mean(np.abs(linear_shap_values), axis=0),
    "mean_shap_logit": np.mean(linear_shap_values, axis=0),
}).merge(
    feature_index[["csv_code", "ref_id", "variable", "survey_year", "feature_group", "question"]],
    on="csv_code",
    how="left",
)
linear_shap_global_df = linear_shap_global_df.sort_values("mean_abs_shap_logit", ascending=False)

local_rows = []
TOP_LOCAL_FEATURES = 8
for row_pos, child_id in enumerate(test_ids):
    top_idx = np.argsort(np.abs(linear_shap_values[row_pos]))[::-1][:TOP_LOCAL_FEATURES]
    for rank, j in enumerate(top_idx, start=1):
        local_rows.append({
            "C0000100": int(child_id),
            "rank": rank,
            "csv_code": feature_cols[j],
            "shap_logit": float(linear_shap_values[row_pos, j]),
            "abs_shap_logit": float(abs(linear_shap_values[row_pos, j])),
            "base_logit": base_logit,
            "predicted_logit": float(predicted_logit[row_pos]),
            "probability": float(logit_test_prob[row_pos]),
            "y_true": int(y_test[row_pos]),
        })
linear_shap_local_df = pd.DataFrame(local_rows).merge(
    feature_index[["csv_code", "ref_id", "variable", "survey_year", "feature_group", "question"]],
    on="csv_code",
    how="left",
)

perm_importance_df.to_csv(OUT_DIR / "l2_logistic_regression_permutation_importance.csv", index=False)
linear_shap_global_df.to_csv(OUT_DIR / "l2_logistic_regression_linear_shap_global_importance.csv", index=False)
linear_shap_local_df.to_csv(OUT_DIR / "l2_logistic_regression_linear_shap_local_top_contributions.csv", index=False)

print("Saved feature importance and XAI outputs:")
print(OUT_DIR / "l2_logistic_regression_coefficients.csv")
print(OUT_DIR / "l2_logistic_regression_permutation_importance.csv")
print(OUT_DIR / "l2_logistic_regression_linear_shap_global_importance.csv")
print(OUT_DIR / "l2_logistic_regression_linear_shap_local_top_contributions.csv")
print("\nTop linear SHAP-style global features:")
linear_shap_global_df.head(15)


## Top 10 Feature Importance Plots

These plots visualize the ten most important features for the logistic regression model using three views: absolute coefficients, permutation AUC drop, and linear SHAP-style global importance.


In [ ]:
import matplotlib.pyplot as plt
import textwrap


def short_label(row, max_width=42):
    year = row.get("survey_year", "")
    var = row.get("variable", row.get("csv_code", ""))
    question = str(row.get("question", ""))
    label = f"{var} ({year}) - {question}"
    return "\n".join(textwrap.wrap(label, width=max_width))


def plot_top10_barh(df, metric, title, filename, signed=False):
    top = df.sort_values(metric, ascending=False).head(10).copy()
    top = top.iloc[::-1]
    labels = [short_label(row) for _, row in top.iterrows()]
    values = top[metric].to_numpy()

    fig, ax = plt.subplots(figsize=(11, 7))
    if signed:
        colors = ["#1b9e77" if v >= 0 else "#d95f02" for v in values]
    else:
        colors = "#4c78a8"
    ax.barh(range(len(top)), values, color=colors)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel(metric)
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUT_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()


coef_plot_df = coef_df.copy()
coef_plot_df["importance"] = coef_plot_df["abs_coefficient"]
plot_top10_barh(
    coef_plot_df,
    "importance",
    "Top 10 Logistic Regression Features by Absolute Coefficient",
    "l2_logistic_regression_top10_coefficients.png",
)

plot_top10_barh(
    perm_importance_df,
    "auc_drop",
    "Top 10 Logistic Regression Features by Permutation Importance (AUC Drop)",
    "l2_logistic_regression_top10_permutation_importance.png",
)

plot_top10_barh(
    linear_shap_global_df,
    "mean_abs_shap_logit",
    "Top 10 Logistic Regression Features by Linear SHAP-style Importance",
    "l2_logistic_regression_top10_linear_shap_importance.png",
)


## SHAP-style XAI Summary Plots

These plots focus only on the linear SHAP-style explanations for logistic regression. The bar plot shows global importance. The dot plot shows the distribution of SHAP-style logit contributions across test cases for the top features.


In [ ]:
# SHAP-style XAI plots for logistic regression.
# These are linear-model SHAP-style values in logit space, computed above as linear_shap_values.

import matplotlib.pyplot as plt
import numpy as np
import textwrap


def shap_label(row, max_width=38):
    year = row.get("survey_year", "")
    var = row.get("variable", row.get("csv_code", ""))
    question = str(row.get("question", ""))
    return "\n".join(textwrap.wrap(f"{var} ({year}) - {question}", width=max_width))


def plot_shap_bar(global_df, metric, title, filename, color="#4c78a8"):
    top = global_df.sort_values(metric, ascending=False).head(10).iloc[::-1].copy()
    labels = [shap_label(row) for _, row in top.iterrows()]
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(range(len(top)), top[metric], color=color)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("mean absolute SHAP-style contribution (logit)")
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUT_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()


def plot_shap_summary_dot(shap_values, feature_values, global_df, title, filename, top_n=10):
    top_codes = global_df.sort_values("mean_abs_shap_logit", ascending=False).head(top_n)["csv_code"].tolist()
    top_indices = [feature_cols.index(code) for code in top_codes]
    label_lookup = global_df.set_index("csv_code").to_dict(orient="index")
    labels = [shap_label(label_lookup[code]) for code in top_codes]

    rng = np.random.default_rng(RANDOM_SEED)
    fig, ax = plt.subplots(figsize=(11, 7))
    all_colors = []
    sc = None
    for y_pos, j in enumerate(top_indices):
        y = np.full(shap_values.shape[0], y_pos, dtype=float) + rng.normal(0, 0.08, shap_values.shape[0])
        values = feature_values[:, j]
        sc = ax.scatter(
            shap_values[:, j],
            y,
            c=values,
            cmap="coolwarm",
            s=12,
            alpha=0.65,
            edgecolors="none",
        )
    ax.axvline(0, color="black", linewidth=1, alpha=0.6)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("SHAP-style contribution to model logit")
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.25)
    if sc is not None:
        cbar = fig.colorbar(sc, ax=ax, pad=0.02)
        cbar.set_label("standardized feature value")
    fig.tight_layout()
    fig.savefig(OUT_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()


plot_shap_bar(
    linear_shap_global_df,
    "mean_abs_shap_logit",
    "Logistic Regression: Top 10 Linear SHAP-style Global Importance",
    "l2_logistic_regression_shap_style_top10_bar.png",
)

plot_shap_summary_dot(
    linear_shap_values,
    x_test_std,
    linear_shap_global_df,
    "Logistic Regression: Linear SHAP-style Summary Plot",
    "l2_logistic_regression_shap_style_summary_dot.png",
)


## How To Read The XAI Outputs

- `l2_logistic_regression_coefficients.csv`: strongest standardized model weights.
- `l2_logistic_regression_permutation_importance.csv`: features that most reduce AUC/accuracy when shuffled.
- `l2_logistic_regression_linear_shap_global_importance.csv`: globally important additive logit contributions.
- `l2_logistic_regression_linear_shap_local_top_contributions.csv`: person-level explanations for individual test cases.

Positive contributions push the model toward `later_persistent_delinquency_contact_2000_2020 = 1`; negative contributions push it toward `0`.
